### Import Library

In [ ]:
import yfinance as yf
import numpy as np
import pandas as pd
from scipy.stats import norm
from scipy.optimize import brentq
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

pio.renderers.default = 'colab'

Data pull (yfinance), math (numpy/scipy), and plotting (plotly) — everything the notebook needs.

### Import Data

In [ ]:
ticker = yf.Ticker("SPY")
S = ticker.history(period="1d")['Close'].iloc[-1]
expiries = ticker.options[:6]

chains = []
for exp in expiries:
    calls = ticker.option_chain(exp).calls
    calls['expirationDate'] = exp
    chains.append(calls)

options_chain = pd.concat(chains, ignore_index=True)

Grabs spot price and the first 6 expiries' call chains. Capped at 6 — far-dated expiries are thin on volume and would just add noise.

### Describe Data

In [ ]:
print(f"Spot: {S:.2f}")
print(f"Contracts pulled: {len(options_chain)}")
display(options_chain.head())
options_chain.info()

Spot: 751.35
Contracts pulled: 715


,contractSymbol,lastTradeDate,strike,lastPrice,bid,ask,change,percentChange,volume,openInterest,impliedVolatility,inTheMoney,contractSize,currency,expirationDate
0,SPY260710C00450000,2026-07-09 19:28:23+00:00,450.0,301.24,301.24,304.02,0.0,0.0,1.0,5,4.912113,True,REGULAR,USD,2026-07-10
1,SPY260710C00480000,2026-06-17 15:11:51+00:00,480.0,269.25,271.18,274.00,0.0,0.0,NaN,0,4.341801,True,REGULAR,USD,2026-07-10
2,SPY260710C00495000,2026-06-09 16:38:21+00:00,495.0,230.27,256.16,258.97,0.0,0.0,NaN,0,4.066411,True,REGULAR,USD,2026-07-10
3,SPY260710C00500000,2026-07-09 18:41:39+00:00,500.0,251.20,251.22,254.03,0.0,0.0,1.0,2,4.011724,True,REGULAR,USD,2026-07-10
4,SPY260710C00520000,2026-06-18 14:43:41+00:00,520.0,227.79,231.16,233.93,0.0,0.0,2.0,2,3.634767,True,REGULAR,USD,2026-07-10


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 715 entries, 0 to 714
Data columns (total 15 columns):
 #   Column             Non-Null Count  Dtype              
---  ------             --------------  -----              
 0   contractSymbol     715 non-null    object             
 1   lastTradeDate      715 non-null    datetime64[ns, UTC]
 2   strike             715 non-null    float64            
 3   lastPrice          715 non-null    float64            
 4   bid                711 non-null    float64            
 5   ask                715 non-null    float64            
 6   change             715 non-null    float64            
 7   percentChange      715 non-null    float64            
 8   volume             667 non-null    float64            
 9   openInterest       715 non-null    int64              
 10  impliedVolatility  715 non-null    float64            
 11  inTheMoney         715 non-null    bool               
 12  contractSize       715 non-null    object         

Sanity check — confirms the pull worked and shows what columns/dtypes you're working with before touching anything.


### Data Visualization

In [ ]:
hist = ticker.history(period="1y")

fig = go.Figure(go.Scatter(x=hist.index, y=hist['Close'], mode='lines', name='SPY'))
fig.update_layout(title="SPY - 1Y", xaxis_title="Date", yaxis_title="Price")
fig.show()

Quick look at the underlying's 1-year trend — context for whether current IVs look elevated relative to recent price action.

In [ ]:
### Data Preprocessing

In [ ]:
df = options_chain[options_chain['volume'] > 0].copy()

df['expirationDate'] = pd.to_datetime(df['expirationDate']).dt.tz_localize(None)
today = pd.Timestamp.today().tz_localize(None)
df['T'] = (df['expirationDate'] - today).dt.days / 365.25
df = df[df['T'] > 0]

df['Moneyness'] = df['strike'] / S
df = df[df['Moneyness'].between(0.85, 1.15)]

r = 0.043
q = 0.013

Drops zero-volume (stale) quotes, computes time-to-maturity in years, drops expired contracts, and restricts to strikes within 15% of spot where liquidity and signal are best. r/q set as flat constants for simplicity.

### Define Target Variable (y) and Feature Variables (X)

X: spot, strike, T, r, q. y: implied vol, backed out per-contract by inverting BSM against the mid price.

Frames the problem: everything except IV is an input; IV is what gets solved for.

### Train Test Split

N/A here — BSM is a closed-form pricer, not something you fit. IV comes from root-finding on the pricing equation, not a train/test split.

No split needed — this isn't a fitted model, it's an inversion of a known formula per contract

### Modeling

In [ ]:
def bs_call(S, K, T, r, q, sigma):
    if sigma <= 0 or T <= 0:
        return max(S * np.exp(-q * T) - K * np.exp(-r * T), 0.0)
    d1 = (np.log(S / K) + (r - q + 0.5 * sigma ** 2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    return S * np.exp(-q * T) * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)

def implied_vol(row):
    K, T = row['strike'], row['T']
    mid = (row['bid'] + row['ask']) / 2 if row['bid'] > 0 and row['ask'] > 0 else row['lastPrice']

    lo = max(S * np.exp(-q * T) - K * np.exp(-r * T), 0.0)
    hi = S * np.exp(-q * T)
    if not (lo < mid < hi):
        return np.nan

    try:
        return brentq(lambda sig: bs_call(S, K, T, r, q, sig) - mid, 1e-6, 5.0, maxiter=200)
    except (ValueError, RuntimeError):
        return np.nan

bs_call prices a European call for a given vol. implied_vol uses mid-price (steadier than last trade) and solves for the vol that reproduces it via brentq. The no-arb bounds check first prevents solving impossible equations.

In [ ]:
df['IV'] = df.apply(implied_vol, axis=1)
df = df.dropna(subset=['IV'])

counts = df['T'].value_counts()
liquid = counts[counts > 4]
front_T = liquid.index.min() if len(liquid) else counts.index[0]
front_month = df[df['T'] == front_T]

fig = make_subplots(rows=1, cols=2, specs=[[{"type": "xy"}, {"type": "scene"}]],
                     subplot_titles=("Front-Month Smile", "Vol Surface"))

fig.add_trace(go.Scatter(x=front_month['Moneyness'], y=front_month['IV'], mode='lines+markers',
                          marker=dict(color=front_month['IV'], colorscale='Viridis', size=8)),
              row=1, col=1)
fig.add_vline(x=1.0, line_dash="dash", line_color="red", row=1, col=1, annotation_text="ATM")

fig.add_trace(go.Mesh3d(x=df['Moneyness'], y=df['T'], z=df['IV'], intensity=df['IV'],
                         colorscale='Viridis', opacity=0.9), row=1, col=2)

fig.update_layout(title=f"SPY Vol Surface | Spot {S:.2f}", height=650,
                   margin=dict(l=20, r=20, b=20, t=60), showlegend=False, hovermode="x unified")
fig.update_xaxes(title_text="Moneyness", range=[0.85, 1.15],
                  rangeslider=dict(visible=True, thickness=0.1), row=1, col=1)
fig.update_yaxes(title_text="IV", row=1, col=1)
fig.layout.scene.update(xaxis_title="Moneyness", yaxis_title="T (yrs)", zaxis_title="IV")

fig.show(config={'scrollZoom': True, 'displayModeBar': True, 'modeBarButtonsToRemove': ['lasso2d', 'select2d']})
fig.write_html("spy_iv_surface.html")

Runs the solver on every row, then plots two views: the front-month smile (skew across strikes at one maturity) and the full 3D surface (skew across both strike and maturity).